# SurfaceTarget-CRC — phase 04 — patch 4 2

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.2
#
# COLORECTAL MODEL IDENTIFICATION AND
# CANDIDATE CRISPR GENE-EFFECT AUDIT
#
# Objectives:
# 1. Validate reconstructed DepMap 26Q3 inputs
# 2. Identify screened colorectal cancer models
# 3. Review colorectal disease and histology annotations
# 4. Audit candidate gene identifier correspondence
# 5. Extract only relevant CRISPR gene-effect columns
# 6. Validate model/sample and numerical integrity
# 7. Calculate descriptive colorectal dependency metrics
# 8. Compare with other screened cancer models
# 9. Review leading research candidates
# 10. Save results and reproducibility metadata
#
# IMPORTANT:
# - No MSS status inferred from tissue origin
# - No automatic gene exclusions
# - No final therapeutic target ranking
# - No causal lineage-specificity claims
# - No modification of original DepMap data
# - Reconstructed input not officially hash-verified
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

import hashlib
import json
import re


ROOT = Path("/content/SurfaceTarget-CRC")

DEPMAP = ROOT / "data/raw/depmap/26Q3"

META = ROOT / "data/metadata"

RESULTS = ROOT / "results/tables"

for directory in [META, RESULTS]:
    directory.mkdir(parents=True, exist_ok=True)


MODEL_PATH = DEPMAP / "Model.csv"

CRISPR_PATH = DEPMAP / "CRISPRGeneEffect.csv"

IMPORT_PATH = (
    META / "depmap_26q3_import_verification.csv"
)

REFERENCE_PATH = (
    META / "depmap_phase4_candidate_reference.csv"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.2")
print("COLORECTAL CRISPR DEPENDENCY AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED RESEARCH INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING DEPMAP INPUT FILES")
print("-" * 65)


required_paths = [
    MODEL_PATH,
    CRISPR_PATH,
    IMPORT_PATH,
    REFERENCE_PATH
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required research file missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


import_audit = pd.read_csv(
    IMPORT_PATH,
    dtype=str,
    keep_default_na=False
)


if import_audit["filename"].duplicated().any():

    raise RuntimeError(
        "Duplicate file entries in import audit."
    )


import_audit = import_audit.set_index(
    "filename"
)


for path in [MODEL_PATH, CRISPR_PATH]:

    if path.name not in import_audit.index:

        raise RuntimeError(
            f"File missing from import audit: {path.name}"
        )

    expected_hash = import_audit.loc[
        path.name,
        "sha256"
    ]

    actual_hash = sha256_file(path)

    if actual_hash != expected_hash:

        raise RuntimeError(
            f"File changed since verification: {path.name}"
        )

    expected_size = int(
        import_audit.loc[
            path.name,
            "file_size_bytes"
        ]
    )

    if path.stat().st_size != expected_size:

        raise RuntimeError(
            f"File size changed: {path.name}"
        )


print("\nDepMap input integrity: PASSED")

print(
    "Source files match the locally recorded "
    "Patch 4.1B checksums."
)

print(
    "Official release checksum comparison: "
    "NOT AVAILABLE"
)


# ------------------------------------------------------------
# 3. LOAD MODEL METADATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING DEPMAP MODEL METADATA")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    dtype=str,
    keep_default_na=False,
    low_memory=False
)


required_model_columns = {
    "ModelID",
    "CellLineName",
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype"
}


if not required_model_columns.issubset(
    models.columns
):

    raise RuntimeError(
        "Required DepMap model annotations missing."
    )


if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in metadata."
    )


models["ModelID"] = (
    models["ModelID"].str.strip()
)


if models["ModelID"].eq("").any():

    raise RuntimeError(
        "Empty model identifiers detected."
    )


print("\nModels in metadata:", len(models))

print("Metadata columns:", len(models.columns))

print("Unique ModelIDs:", models["ModelID"].nunique())


# ------------------------------------------------------------
# 4. LOAD SCREENED MODEL IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IDENTIFYING MODELS WITH CRISPR MEASUREMENTS")
print("-" * 65)


# The reconstructed CRISPR CSV retains
# its original blank first-column header.
#
# Read only the first column initially.
# No genome-wide matrix is loaded here.


crispr_ids = pd.read_csv(
    CRISPR_PATH,
    usecols=[0],
    dtype=str,
    low_memory=False
)


crispr_ids.columns = ["ModelID"]


crispr_ids["ModelID"] = (
    crispr_ids["ModelID"]
    .astype(str)
    .str.strip()
)


if crispr_ids["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model identifiers."
    )


if crispr_ids["ModelID"].eq("").any():

    raise RuntimeError(
        "Missing CRISPR model identifiers."
    )


screened_ids = set(
    crispr_ids["ModelID"]
)


metadata_ids = set(
    models["ModelID"]
)


if not screened_ids.issubset(metadata_ids):

    raise RuntimeError(
        "Some CRISPR models lack model metadata."
    )


screened_models = models[
    models["ModelID"].isin(
        screened_ids
    )
].copy()


print("\nModels with CRISPR measurements:",
      len(screened_ids))

print(
    "Screened models matching metadata:",
    len(screened_models)
)


if len(screened_ids) != 1361:

    raise RuntimeError(
        "Unexpected screened model count. "
        "Expected 1,361 from Patch 4.1B."
    )


print("\nModel correspondence: PASSED")


# ------------------------------------------------------------
# 5. EXPLORE COLORECTAL MODEL ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COLORECTAL MODEL ANNOTATION DISCOVERY")
print("-" * 65)


# Do not assume all Bowel-lineage models
# are colorectal adenocarcinoma models.
#
# First inspect disease classifications
# using the actual release metadata.


bowel_mask = (
    screened_models["OncotreeLineage"]
    .str.contains(
        r"bowel|colorectal",
        case=False,
        regex=True,
        na=False
    )
)


bowel_models = screened_models[
    bowel_mask
].copy()


print("\nScreened models in bowel-related lineages:",
      len(bowel_models))


print("\nPrimary disease labels in bowel-related models:")


print(
    bowel_models[
        "OncotreePrimaryDisease"
    ]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 6. IDENTIFY COLORECTAL CANCER MODELS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTING COLORECTAL CANCER MODELS")
print("-" * 65)


# The primary disease annotation must
# explicitly identify colon, rectal,
# or colorectal malignancy.
#
# We do not include every bowel-lineage
# model automatically.


colorectal_mask = (
    screened_models["OncotreePrimaryDisease"]
    .str.contains(
        r"colorectal|colon|rectal",
        case=False,
        regex=True,
        na=False
    )
)


colorectal_models = screened_models[
    colorectal_mask
].copy()


print("\nScreened colorectal models:",
      len(colorectal_models))


print("\nColorectal primary disease distribution:")

print(
    colorectal_models[
        "OncotreePrimaryDisease"
    ]
    .value_counts()
    .to_string()
)


print("\nColorectal subtype distribution:")

print(
    colorectal_models[
        "OncotreeSubtype"
    ]
    .replace("", "NOT AVAILABLE")
    .value_counts()
    .to_string()
)


if colorectal_models.empty:

    raise RuntimeError(
        "No screened colorectal models identified. "
        "Inspect the printed OncoTree annotations "
        "before modifying selection criteria."
    )


# ------------------------------------------------------------
# 6A. Review unusual colorectal histologies
# ------------------------------------------------------------

# Most of our patient-tissue discovery analysis
# concerns colorectal adenocarcinoma.
#
# Flag clearly distinct histologies for review
# rather than automatically treating them
# as representative adenocarcinoma models.


ATYPICAL_HISTOLOGY_PATTERN = (
    r"neuroendocrine|small.cell|"
    r"sarcoma|lymphoma|gastrointestinal.stromal"
)


colorectal_models["histology_review_flag"] = (
    colorectal_models["OncotreeSubtype"]
    .str.contains(
        ATYPICAL_HISTOLOGY_PATTERN,
        case=False,
        regex=True,
        na=False
    )
)


histology_review = colorectal_models[
    colorectal_models["histology_review_flag"]
].copy()


print("\nModels requiring distinct-histology review:",
      len(histology_review))


if not histology_review.empty:

    print(
        histology_review[
            [
                "ModelID",
                "CellLineName",
                "OncotreePrimaryDisease",
                "OncotreeSubtype"
            ]
        ].to_string(index=False)
    )


# Use the main colorectal group while
# withholding clearly distinct histologies
# from the primary descriptive summaries.
#
# All reviewed models are preserved in
# the full colorectal model inventory.


primary_crc_models = colorectal_models[
    ~colorectal_models["histology_review_flag"]
].copy()


print("\nPrimary colorectal analysis models:",
      len(primary_crc_models))


if len(primary_crc_models) < 3:

    raise RuntimeError(
        "Insufficient colorectal models "
        "for descriptive analysis."
    )


# ------------------------------------------------------------
# 6B. Check possible patient-level duplication
# ------------------------------------------------------------

if "PatientID" in primary_crc_models.columns:

    patient_values = (
        primary_crc_models["PatientID"]
        .replace("", np.nan)
        .dropna()
    )

    repeated_patients = int(
        patient_values.duplicated().sum()
    )

else:

    repeated_patients = np.nan


print("\nRepeated nonmissing PatientIDs:",
      repeated_patients)


print(
    "Multiple models from one patient, when present, "
    "must not be described as independent patients."
)


# ------------------------------------------------------------
# 6C. Inspect possible MSI annotations
# ------------------------------------------------------------

msi_related_columns = [
    column
    for column in models.columns
    if re.search(
        r"msi|microsatellite|mantis",
        column,
        flags=re.IGNORECASE
    )
]


print("\nPotential MSI annotation columns:")

print(
    msi_related_columns
    if msi_related_columns
    else "None identified in Model.csv"
)


print(
    "\nNo models have been assigned MSS status."
)


# ------------------------------------------------------------
# 7. LOAD THE PHASE 4 CANDIDATE REFERENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CANDIDATE GENE REFERENCE")
print("-" * 65)


reference = pd.read_csv(
    REFERENCE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str,
        "gene_name": str
    },
    low_memory=False
)


if len(reference) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if reference["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate transcriptomic gene IDs."
    )


if reference["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate gene symbols."
    )


reference["gene_name"] = (
    reference["gene_name"]
    .astype(str)
    .str.strip()
)


reference["protein_enriched"] = (
    reference["protein_enriched"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)


if int(reference["protein_enriched"].sum()) != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


print("\nCandidate genes:", len(reference))

print(
    "Cross-omics-supported genes:",
    int(reference["protein_enriched"].sum())
)


# ------------------------------------------------------------
# 8. INSPECT CRISPR GENE IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CRISPR GENE IDENTIFIER AUDIT")
print("-" * 65)


# DepMap headers have the general format:
#
# A1BG (1)
# CDH3 (1001)
#
# Gene symbol and Entrez ID are preserved.
#
# Matching candidate gene symbols is an
# initial correspondence rule, not a
# fully verified Ensembl-to-Entrez crosswalk.


crispr_header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


print("\nTotal CRISPR columns:",
      len(crispr_header))


if len(crispr_header) != 18435:

    raise RuntimeError(
        "Unexpected CRISPR column count."
    )


model_column = crispr_header[0]


if model_column not in [
    "Unnamed: 0",
    "ModelID"
]:

    raise RuntimeError(
        "Unexpected CRISPR model column."
    )


GENE_PATTERN = re.compile(
    r"^(.*?) \((\d+)\)$"
)


gene_header_records = []

for column in crispr_header[1:]:

    match = GENE_PATTERN.fullmatch(
        column
    )

    if match:

        gene_symbol = match.group(1)

        entrez_id = match.group(2)

    else:

        gene_symbol = None

        entrez_id = None

    gene_header_records.append({

        "depmap_column": column,

        "depmap_gene_symbol": gene_symbol,

        "depmap_entrez_id": entrez_id,

        "header_parsed": match is not None
    })


gene_headers = pd.DataFrame(
    gene_header_records
)


parsed_count = int(
    gene_headers["header_parsed"].sum()
)


print("\nCRISPR gene columns:",
      len(gene_headers))

print(
    "Headers matching symbol/Entrez format:",
    parsed_count
)


if parsed_count < 18000:

    raise RuntimeError(
        "Unexpected DepMap gene header format."
    )


# ------------------------------------------------------------
# 9. MAP CANDIDATES TO CRISPR GENE COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MATCHING RESEARCH CANDIDATES TO DEPMAP")
print("-" * 65)


gene_matches = {}

for symbol, group in gene_headers.groupby(
    "depmap_gene_symbol",
    dropna=True
):

    gene_matches[symbol] = group.to_dict(
        orient="records"
    )


mapping_records = []


for _, row in reference.iterrows():

    symbol = row["gene_name"]

    matches = gene_matches.get(
        symbol,
        []
    )

    if len(matches) == 1:

        mapping_status = (
            "Unique exact gene-symbol match"
        )

        selected_column = matches[0][
            "depmap_column"
        ]

        entrez_id = matches[0][
            "depmap_entrez_id"
        ]

    elif len(matches) > 1:

        mapping_status = (
            "Ambiguous gene-symbol match"
        )

        selected_column = None
        entrez_id = None

    else:

        mapping_status = (
            "No exact gene-symbol match"
        )

        selected_column = None
        entrez_id = None


    mapping_records.append({

        "gene_id": row["gene_id"],

        "ensembl_stable_id":
            row["ensembl_stable_id"],

        "gene_name": symbol,

        "protein_enriched":
            bool(row["protein_enriched"]),

        "depmap_matching_columns":
            len(matches),

        "depmap_mapping_status":
            mapping_status,

        "depmap_column":
            selected_column,

        "depmap_entrez_id":
            entrez_id
    })


gene_mapping = pd.DataFrame(
    mapping_records
)


print("\nMapping status distribution:")

print(
    gene_mapping[
        "depmap_mapping_status"
    ]
    .value_counts()
    .to_string()
)


uniquely_mapped = gene_mapping[
    gene_mapping["depmap_mapping_status"]
    .eq("Unique exact gene-symbol match")
].copy()


print("\nUniquely matched candidates:",
      len(uniquely_mapped))


unresolved = gene_mapping[
    ~gene_mapping["depmap_mapping_status"]
    .eq("Unique exact gene-symbol match")
].copy()


print("\nUnresolved candidate mappings:",
      len(unresolved))


if not unresolved.empty:

    print("\nFirst 25 unresolved genes:")

    print(
        unresolved[
            [
                "gene_name",
                "depmap_mapping_status"
            ]
        ]
        .head(25)
        .to_string(index=False)
    )


if len(uniquely_mapped) < 500:

    raise RuntimeError(
        "Unexpectedly low candidate gene coverage. "
        "Review identifier correspondence."
    )


# ------------------------------------------------------------
# 10. EXTRACT CANDIDATE CRISPR GENE EFFECTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING CANDIDATE CRISPR MEASUREMENTS")
print("-" * 65)


selected_gene_columns = (
    uniquely_mapped["depmap_column"]
    .tolist()
)


columns_to_read = (
    [model_column]
    + selected_gene_columns
)


# Read only the candidate gene columns.
#
# This avoids loading all 18,434 gene
# columns into memory.


effects = pd.read_csv(
    CRISPR_PATH,
    usecols=columns_to_read,
    dtype={model_column: str},
    low_memory=False
)


effects = effects.rename(
    columns={
        model_column: "ModelID"
    }
)


effects["ModelID"] = (
    effects["ModelID"]
    .astype(str)
    .str.strip()
)


if effects["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model rows."
    )


effects = effects.set_index(
    "ModelID"
)


if set(effects.index) != screened_ids:

    raise RuntimeError(
        "CRISPR model inventory changed "
        "during candidate extraction."
    )


if set(effects.columns) != set(
    selected_gene_columns
):

    raise RuntimeError(
        "Extracted CRISPR gene columns differ "
        "from the validated mapping."
    )


# Convert only extracted gene-effect columns
# to numeric measurements.


effects = effects.apply(
    pd.to_numeric,
    errors="raise"
)


if np.isinf(
    effects.to_numpy(dtype=float)
).any():

    raise RuntimeError(
        "Infinite CRISPR gene-effect values detected."
    )


print("\nCandidate CRISPR matrix extracted.")

print("Matrix dimensions:", effects.shape)

print(
    "Models represented:",
    effects.index.nunique()
)

print(
    "Gene columns represented:",
    len(effects.columns)
)


print("\nCandidate matrix validation: PASSED")


# ------------------------------------------------------------
# 11. CREATE COLORECTAL AND COMPARATOR MATRICES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING FUNCTIONAL ANALYSIS GROUPS")
print("-" * 65)


crc_ids = (
    primary_crc_models["ModelID"]
    .tolist()
)


crc_effects = effects.loc[
    crc_ids
].copy()


other_effects = effects.drop(
    index=crc_ids
).copy()


print("\nColorectal models:",
      len(crc_effects))

print("Other screened models:",
      len(other_effects))


if len(crc_effects) < 3:

    raise RuntimeError(
        "Insufficient colorectal CRISPR observations."
    )


if len(other_effects) < 50:

    raise RuntimeError(
        "Insufficient comparator models."
    )


print("\nAnalysis groups: PASSED")


# ------------------------------------------------------------
# 12. CALCULATE DESCRIPTIVE GENE-EFFECT STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING COLORECTAL CRISPR GENE EFFECTS")
print("-" * 65)


def summarize_gene_effect(
    matrix,
    gene_column
):

    values = (
        matrix[gene_column]
        .dropna()
        .to_numpy(dtype=float)
    )

    n = len(values)

    if n == 0:

        return {
            "n": 0,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "fraction_below_minus_05": np.nan,
            "fraction_below_minus_1": np.nan
        }

    return {

        "n": n,

        "median": float(
            np.median(values)
        ),

        "q25": float(
            np.percentile(values, 25)
        ),

        "q75": float(
            np.percentile(values, 75)
        ),

        "fraction_below_minus_05": float(
            np.mean(values < -0.5)
        ),

        "fraction_below_minus_1": float(
            np.mean(values < -1.0)
        )

    }


summary_records = []


for _, row in gene_mapping.iterrows():

    gene_name = row["gene_name"]

    gene_column = row["depmap_column"]

    record = row.to_dict()

    if pd.isna(gene_column):

        record.update({

            "crc_models_measured": 0,

            "crc_median_gene_effect": np.nan,

            "crc_q25_gene_effect": np.nan,

            "crc_q75_gene_effect": np.nan,

            "crc_fraction_below_minus_05": np.nan,

            "crc_fraction_below_minus_1": np.nan,

            "other_models_measured": 0,

            "other_median_gene_effect": np.nan,

            "other_fraction_below_minus_05": np.nan,

            "crc_minus_other_median_effect": np.nan
        })

    else:

        crc_stats = summarize_gene_effect(
            crc_effects,
            gene_column
        )

        other_stats = summarize_gene_effect(
            other_effects,
            gene_column
        )

        if (
            crc_stats["n"] > 0
            and other_stats["n"] > 0
        ):

            median_difference = (
                crc_stats["median"]
                - other_stats["median"]
            )

        else:

            median_difference = np.nan

        record.update({

            "crc_models_measured":
                crc_stats["n"],

            "crc_median_gene_effect":
                crc_stats["median"],

            "crc_q25_gene_effect":
                crc_stats["q25"],

            "crc_q75_gene_effect":
                crc_stats["q75"],

            "crc_fraction_below_minus_05":
                crc_stats[
                    "fraction_below_minus_05"
                ],

            "crc_fraction_below_minus_1":
                crc_stats[
                    "fraction_below_minus_1"
                ],

            "other_models_measured":
                other_stats["n"],

            "other_median_gene_effect":
                other_stats["median"],

            "other_fraction_below_minus_05":
                other_stats[
                    "fraction_below_minus_05"
                ],

            "crc_minus_other_median_effect":
                median_difference
        })


    summary_records.append(record)


dependency_summary = pd.DataFrame(
    summary_records
)


if len(dependency_summary) != 948:

    raise RuntimeError(
        "Dependency summary changed candidate count."
    )


print("\nCandidate dependency summaries:",
      len(dependency_summary))


print(
    "Candidates with colorectal CRISPR data:",
    int(
        dependency_summary[
            "crc_models_measured"
        ].gt(0).sum()
    )
)


print(
    "Candidates with >=5 colorectal measurements:",
    int(
        dependency_summary[
            "crc_models_measured"
        ].ge(5).sum()
    )
)


# ------------------------------------------------------------
# 13. EXPLORE STRONGER NEGATIVE GENE EFFECTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXPLORATORY COLORECTAL FITNESS EFFECTS")
print("-" * 65)


# Descriptive screening only.
#
# No statistical claims about selective
# dependency are made here.


rankable = dependency_summary[
    dependency_summary[
        "crc_models_measured"
    ].ge(5)
].copy()


rankable = rankable.sort_values(
    "crc_median_gene_effect",
    ascending=True
)


print("\nTop 20 candidates by most negative "
      "median colorectal gene effect:")


print(
    rankable[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "crc_minus_other_median_effect"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 14. REVIEW LEADING CANCER-TARGET CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE FUNCTIONAL REVIEW")
print("-" * 65)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


leading = dependency_summary[
    dependency_summary["gene_name"]
    .isin(LEADING_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


print("\n")

print(
    leading[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "crc_minus_other_median_effect"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\nIMPORTANT:")

print(
    "CRISPR dependency is not necessary "
    "for antibody-drug conjugate targeting."
)

print(
    "Weak dependency in cancer cell lines "
    "does not exclude stromal or "
    "antigen-mediated targeting strategies."
)


# ------------------------------------------------------------
# 15. SAVE COLORECTAL MODEL INVENTORIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING DEPMAP MODEL INVENTORIES")
print("-" * 65)


screened_models.to_csv(
    RESULTS /
    "depmap_26q3_screened_model_metadata.csv",
    index=False
)


colorectal_models.to_csv(
    RESULTS /
    "depmap_26q3_colorectal_model_inventory.csv",
    index=False
)


primary_crc_models.to_csv(
    RESULTS /
    "depmap_26q3_primary_crc_analysis_models.csv",
    index=False
)


histology_review.to_csv(
    RESULTS /
    "depmap_26q3_histology_review_models.csv",
    index=False
)


# ------------------------------------------------------------
# 16. SAVE CANDIDATE GENE MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CRISPR GENE MAPPINGS")
print("-" * 65)


gene_mapping.to_csv(
    RESULTS /
    "depmap_26q3_candidate_gene_mapping.csv",
    index=False
)


unresolved.to_csv(
    RESULTS /
    "depmap_26q3_unresolved_gene_mapping.csv",
    index=False
)


# ------------------------------------------------------------
# 17. SAVE COLORECTAL CANDIDATE MATRIX
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CANDIDATE CRISPR MATRIX")
print("-" * 65)


column_to_symbol = dict(
    zip(
        uniquely_mapped["depmap_column"],
        uniquely_mapped["gene_name"]
    )
)


crc_matrix_export = crc_effects.rename(
    columns=column_to_symbol
).copy()


crc_matrix_export.index.name = "ModelID"


crc_matrix_export.to_csv(
    RESULTS /
    "depmap_26q3_crc_candidate_gene_effects.csv"
)


dependency_summary.to_csv(
    RESULTS /
    "depmap_26q3_candidate_gene_effect_summary.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "depmap_26q3_leading_candidate_dependency.csv",
    index=False
)


print("\nAll dependency audit tables saved.")


# ------------------------------------------------------------
# 18. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PHASE 4 PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.2",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "depmap_release": "26Q3",

    "model_source": str(MODEL_PATH),

    "crispr_source": str(CRISPR_PATH),

    "model_sha256": sha256_file(
        MODEL_PATH
    ),

    "crispr_sha256": sha256_file(
        CRISPR_PATH
    ),

    "source_integrity": (
        "Locally checksum-verified reconstructed "
        "CSV; official source checksum unavailable"
    ),

    "screened_models": int(
        len(screened_ids)
    ),

    "colorectal_models_identified": int(
        len(colorectal_models)
    ),

    "primary_crc_models": int(
        len(primary_crc_models)
    ),

    "histology_review_models": int(
        len(histology_review)
    ),

    "candidate_genes": int(
        len(gene_mapping)
    ),

    "unique_exact_symbol_matches": int(
        len(uniquely_mapped)
    ),

    "unresolved_gene_mappings": int(
        len(unresolved)
    ),

    "candidate_gene_matching": (
        "Unique exact gene-symbol correspondence "
        "to DepMap Symbol (EntrezID) headers; "
        "Entrez ID retained for future crosswalk validation"
    ),

    "model_selection": (
        "Screened models with primary disease "
        "explicitly annotated as colorectal, "
        "colon or rectal; distinct histologies "
        "separately flagged"
    ),

    "msi_status_assigned": False,

    "gene_effect_summary": [
        "Median",
        "25th percentile",
        "75th percentile",
        "Fraction below -0.5",
        "Fraction below -1",
        "Other-model median",
        "Difference between group medians"
    ],

    "inferential_dependency_testing":
        False,

    "target_ranking_performed": False,

    "candidate_exclusions": 0,

    "limitations": [

        "The CRISPR file was reconstructed "
        "from upload fragments.",

        "Official byte-for-byte checksum "
        "verification is unavailable.",

        "Colorectal model status does not "
        "establish MSS status.",

        "Different colorectal histologies "
        "require biological review.",

        "DepMap models may not be independent "
        "at the patient level.",

        "Initial gene mapping uses exact "
        "symbols rather than validated "
        "Ensembl-to-Entrez correspondence.",

        "More negative gene-effect values "
        "suggest stronger knockout fitness effects.",

        "Median differences are descriptive "
        "and do not establish lineage selectivity.",

        "The noncolorectal comparator includes "
        "heterogeneous cancer lineages.",

        "CRISPR dependency is not a necessary "
        "criterion for ADC antigen suitability.",

        "Stromal antigens may not show dependency "
        "in cancer-cell-intrinsic screens.",

        "No therapeutic targets have been validated."
    ]
}


with open(
    META /
    "depmap_26q3_candidate_dependency_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.2 SUMMARY")
print("=" * 65)


print("\nDepMap release: Public 26Q3")


print(
    "Screened cancer models:",
    len(screened_ids)
)


print(
    "Colorectal models identified:",
    len(colorectal_models)
)


print(
    "Primary colorectal analysis models:",
    len(primary_crc_models)
)


print(
    "Distinct-histology models flagged:",
    len(histology_review)
)


print(
    "\nTranscriptomic screening genes:",
    len(gene_mapping)
)


print(
    "Uniquely matched DepMap genes:",
    len(uniquely_mapped)
)


print(
    "Unresolved gene mappings:",
    len(unresolved)
)


print(
    "Genes with available colorectal CRISPR data:",
    int(
        dependency_summary[
            "crc_models_measured"
        ].gt(0).sum()
    )
)


print("\nSaved outputs:")

print("1. depmap_26q3_screened_model_metadata.csv")

print("2. depmap_26q3_colorectal_model_inventory.csv")

print("3. depmap_26q3_primary_crc_analysis_models.csv")

print("4. depmap_26q3_histology_review_models.csv")

print("5. depmap_26q3_candidate_gene_mapping.csv")

print("6. depmap_26q3_unresolved_gene_mapping.csv")

print("7. depmap_26q3_crc_candidate_gene_effects.csv")

print("8. depmap_26q3_candidate_gene_effect_summary.csv")

print("9. depmap_26q3_leading_candidate_dependency.csv")

print("10. depmap_26q3_candidate_dependency_provenance.json")


print("\nModel matching: PASSED")

print("Gene identifier audit: COMPLETE")

print("Descriptive CRISPR analysis: COMPLETE")


print("\nNo MSS status inferred.")

print("No final target scores assigned.")

print("No source files modified.")

print("No candidate genes excluded.")


print("\nPATCH 4.2 COMPLETE")

print("=" * 65)